In [ ]:
# Step 1: Mount Google Drive and import libraries
from google.colab import drive
import os
import zipfile
import shutil
import glob
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Step 2: Define paths and extract dataset
zip_path = '/content/drive/My Drive/Drone detection dataset.zip'
extract_path = '/content/drone_detection_dataset'

# Create extraction directory
os.makedirs(extract_path, exist_ok=True)

# Extract the zip file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Dataset extracted successfully to:", extract_path)

✅ Dataset extracted successfully to: /content/drone_detection_dataset


In [ ]:
folders = ['train', 'valid', 'test']
base_path = extract_path  # e.g., /content/drone_detection_dataset

In [ ]:
# === STEP 3: Clean labels: Keep only drone (class 2), delete others ===
for folder in folders:
    labels_dir = os.path.join(base_path, folder, 'labels')
    label_files = glob.glob(os.path.join(labels_dir, '*.txt'))

    for file in label_files:
        with open(file, 'r') as f:
            lines = f.readlines()

        # Keep only lines that start with "2 " (drone), remove class 0/1
        new_lines = [line for line in lines if line.startswith('2 ')]

        # If no drone lines left → make empty file (background)
        if not new_lines:
            open(file, 'w').close()  # Create empty file
        else:
            # Remap "2 x y w h" → "0 x y w h"
            fixed_lines = ['0' + line[1:] for line in new_lines]
            with open(file, 'w') as f:
                f.writelines(fixed_lines)

print("✅ Non-drone labels removed. Drone class 2 → 0 remapped.")

✅ Non-drone labels removed. Drone class 2 → 0 remapped.


In [ ]:
yaml_content = """train: {}/train/images
val: {}/valid/images
test: {}/test/images
names:
  0: drone
nc: 1
""".format(base_path, base_path, base_path)

with open(os.path.join(base_path, 'data.yaml'), 'w') as f:
    f.write(yaml_content)

print("✅ data.yaml created at:", os.path.join(base_path, 'data.yaml'))
print("🔥 ALL SET! You can now train with YOLO.")

✅ data.yaml created at: /content/drone_detection_dataset/data.yaml
🔥 ALL SET! You can now train with YOLO.


In [ ]:
# 🔁 Copy cleaned dataset to Drive (SAVE IT!)
# import shutil

In [ ]:
# Download to Driver edited dataset ------------------------------------------------------

# source = '/content/drone_detection_dataset'
# destination = '/content/drive/My Drive/drone_detection_dataset'

# # Create destination folder if it doesn't exist
# os.makedirs(os.path.dirname(destination), exist_ok=True)

# # Copy entire folder
# shutil.copytree(source, destination, dirs_exist_ok=True)

# print("✅ Cleaned dataset saved to Drive:", destination)

In [ ]:
!pip install ultralytics --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 30.0 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
!nvidia-smi

Sat Aug 30 10:36:12 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# === FIRST SESSION ===
# Start fresh with a small/faster model
model = YOLO('yolo11n.pt')  # nano is fastest; change to 'yolo11s.pt' if you want more accuracy

# Train with optimized settings for FREE Colab (T4 GPU)
results = model.train(
    # 🔧 Dataset
    data='/content/drive/MyDrive/drone_detection_dataset/data.yaml',  # Path to your data.yaml

    # 🖼️ Image size (smaller = faster)
    imgsz=320,           # 320x320 (half of 640) → 2x faster on GPU

    # 📦 Batch size (safe for free Colab T4)
    batch=16,            # 16 is safe; 32 may crash on free tier

    # ⏱️ Training
    epochs=30,           # Total epochs
    device=0,            # Use GPU (T4)

    # 💾 Save to Drive (permanent)
    project='/content/drive/MyDrive/drone_detection_run2',
    name='train_fast_320',  # Folder name
    exist_ok=True,       # Overwrite if exists

    # 🛠️ Optimization for free Colab
    workers=2,           # Low CPU workers (saves RAM)
    patience=10,         # Stop early if no improvement
    cache=False,         # Don't cache images (saves RAM)
    close_mosaic=10      # Helps with small datasets
)

NameError: name 'YOLO' is not defined

In [ ]:
model = YOLO('/content/drive/MyDrive/drone_detection_run2/train_fast_320/weights/last.pt')

# Resume training with same settings
results = model.train(
    data='/content/drone_detection_dataset/data.yaml',
    imgsz=320,
    batch=16,
    epochs=30,  # Total target epochs; YOLO will auto-resume from last
    device='cpu',
    project='/content/drive/MyDrive/drone_detection_run2',
    name='train_fast_320',
    exist_ok=True,
    workers=2,
    patience=10,
    cache=False,
    close_mosaic=10,
    resume=True  # 👈 This tells YOLO to continue from last.pt
)

Ultralytics 8.3.189 🚀 Python-3.12.11 torch-2.8.0+cu126 CPU (Intel Xeon 2.20GHz)


PermissionError: [Errno 1] Operation not permitted: '/content/drive/MyDrive/drone_detection_run2/train_fast_320/args.yaml'